# Decision Layer — Record layer + logging/feedback loop

Reproducible demonstration of the **Record** step of the Record → Decide → Act architecture (Exhibit 14 §9) and its production logging extension. Previously the RedMap backend was a *stateless scorer* (Exhibit 14 §3). Now:

1. **every model verdict** (`/identity/score`, `/ics/score`, `/map`) is persisted to a durable SQLite trail, enriched with request metadata (latency, status, client, path);
2. **every scored response returns an `X-Verdict-Id` header** so any live client can refer back to that exact decision;
3. **ground truth is attached via a feedback channel** (`POST /decision/verdicts/{id}/feedback`) — decoupled from scoring, as in a real deployment where the true label arrives later from an analyst or downstream system;
4. **live precision/recall/specificity** are computed from the DB (`GET /decision/metrics`), persisting what the Exhibit 14 tally did from text logs.

The notebook runs the real backend in-process (FastAPI `TestClient`); it is self-contained and deterministic.

In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'backend'))
# Fresh demo DB so the run is deterministic and never touches the real trail.
DEMO_DB = ROOT / 'data' / 'verdicts_demo.db'
if DEMO_DB.exists(): DEMO_DB.unlink()
os.environ['VERDICT_DB'] = str(DEMO_DB)
print('repo root:', ROOT, '\ndemo DB  :', DEMO_DB)

repo root: C:\Users\User\ai-cybersecurity-portfolio 
demo DB  : C:\Users\User\ai-cybersecurity-portfolio\data\verdicts_demo.db


## 1. Start the backend in-process
Loading `app` initializes all three models and encodes the HIPAA corpus, so this first cell may take a little time.

In [2]:
from fastapi.testclient import TestClient
import app as appmod
client = TestClient(appmod.app)
print('health:', client.get('/health').json())

C:\Users\User\ai-cybersecurity-portfolio\venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa
C:\Users\User\ai-cybersecurity-portfolio\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8807.43it/s]

health: {'status': 'ok'}


## 2. Score events — and capture the `X-Verdict-Id` header
This header is how a live client refers back to a specific decision to label it later.

In [3]:
def score_identity(user, pc, auth, logon, succ):
    r = client.post('/identity/score', json={'src_user':user,'src_pc':pc,'auth_type':auth,
        'logon_type':logon,'orientation':'LogOn','success':succ})
    return r.headers['X-Verdict-Id'], r.json()['is_anomaly']

vid_norm, a1 = score_identity('U1006@DOM1','C1028','Negotiate','Network','Success')
vid_susp, a2 = score_identity('guest@DOM1','C1047','?','RemoteInteractive','Fail')

ex = client.get('/ics/example').json()
r_icsn = client.post('/ics/score', json={'readings': ex}); vid_icsn = r_icsn.headers['X-Verdict-Id']
spiked = dict(ex); spiked['P1_LIT01'] = 5000.0
r_icsa = client.post('/ics/score', json={'readings': spiked}); vid_icsa = r_icsa.headers['X-Verdict-Id']

print('identity normal  -> verdict', vid_norm, '| is_anomaly', a1)
print('identity suspect -> verdict', vid_susp, '| is_anomaly', a2)
print('ics normal  -> verdict', vid_icsn, '| error', r_icsn.json()['reconstruction_error'])
print('ics attack  -> verdict', vid_icsa, '| error', r_icsa.json()['reconstruction_error'])

identity normal  -> verdict 1 | is_anomaly False
identity suspect -> verdict 2 | is_anomaly False
ics normal  -> verdict 3 | error 0.001583
ics attack  -> verdict 4 | error 359.190002


## 3. The trail, enriched with request metadata
Each verdict row now also carries the latency, HTTP status, client, and path captured by the app middleware — logged for every request the live system handles.

In [4]:
import pandas as pd
cols = ['id','model','subject','flagged','score','latency_ms','status','client','path','ground_truth']
pd.DataFrame(client.get('/decision/verdicts?limit=20').json())[cols]

,id,model,subject,flagged,score,latency_ms,status,client,path,ground_truth
0,4,ics,NaN,True,359.190002,7.175,200,testclient,/ics/score,None
1,3,ics,NaN,False,0.001583,9.116,200,testclient,/ics/score,None
2,2,identity,guest@DOM1,False,0.010572,19.919,200,testclient,/identity/score,None
3,1,identity,U1006@DOM1,False,0.184991,31.905,200,testclient,/identity/score,None


## 4. Attach ground truth via feedback (the production mechanism)
A live client — analyst UI, SOAR, ticket-resolution webhook, or (here) our test harness — labels a decision by its `X-Verdict-Id`. Labels are decoupled from scoring and vocabulary-normalized (`attack`/`normal` → `malicious`/`benign`).

In [5]:
for vid, label in [(vid_norm,'normal'), (vid_susp,'attack'), (vid_icsn,'benign'), (vid_icsa,'malicious')]:
    resp = client.post(f'/decision/verdicts/{vid}/feedback', json={'ground_truth': label})
    print(resp.json())

{'verdict_id': 1, 'ground_truth': 'normal', 'recorded': True}
{'verdict_id': 2, 'ground_truth': 'attack', 'recorded': True}


{'verdict_id': 3, 'ground_truth': 'benign', 'recorded': True}
{'verdict_id': 4, 'ground_truth': 'malicious', 'recorded': True}


## 5. Live detection metrics, computed from the DB
Now that decisions carry ground truth, precision/recall/specificity are computed over the labelled subset — the confusion matrix the Exhibit 14 tally produced, but persisted and queryable in real time. (Recall < 1 here reflects the single-event 'suspect' login, which is correctly *not* a behavioural burst — Exhibit 14: breadth of access across many machines is the signal.)

In [6]:
print('overall :', client.get('/decision/metrics').json())
print('identity:', client.get('/decision/metrics?model=identity').json())
print('ics     :', client.get('/decision/metrics?model=ics').json())
print('coverage:', client.get('/decision/stats').json())

overall : {'model': 'all', 'labeled': 4, 'tp': 1, 'fp': 0, 'fn': 1, 'tn': 2, 'precision': 1.0, 'recall': 0.5, 'specificity': 1.0, 'accuracy': 0.75}
identity: {'model': 'identity', 'labeled': 2, 'tp': 0, 'fp': 0, 'fn': 1, 'tn': 1, 'precision': None, 'recall': 0.0, 'specificity': 1.0, 'accuracy': 0.5}
ics     : {'model': 'ics', 'labeled': 2, 'tp': 1, 'fp': 0, 'fn': 0, 'tn': 1, 'precision': 1.0, 'recall': 1.0, 'specificity': 1.0, 'accuracy': 1.0}
coverage: {'verdicts': 4, 'flagged': 1, 'labeled': 4, 'audited_requests': 10, 'by_model': {'ics': {'count': 2, 'flagged': 1}, 'identity': {'count': 2, 'flagged': 0}}, 'db_path': 'C:\\Users\\User\\ai-cybersecurity-portfolio\\data\\verdicts_demo.db'}


## 6. Complete request audit (non-scored traffic)
Requests that produce no verdict — health checks, validation failures, unknown ids — are logged too, so the record of what the live system handled is complete.

In [7]:
client.get('/health'); client.post('/decision/verdicts/99999/feedback', json={'ground_truth':'benign'})
pd.DataFrame(client.get('/decision/requests?limit=10').json())[['method','path','status','latency_ms']]

,method,path,status,latency_ms
0,POST,/decision/verdicts/99999/feedback,404,1.232
1,GET,/health,200,0.937
2,GET,/decision/stats,200,1.550
3,GET,/decision/metrics,200,1.038
4,GET,/decision/metrics,200,1.196
5,GET,/decision/metrics,200,1.320
6,POST,/decision/verdicts/4/feedback,200,5.882
7,POST,/decision/verdicts/3/feedback,200,5.651
8,POST,/decision/verdicts/2/feedback,200,6.442
9,POST,/decision/verdicts/1/feedback,200,6.891


## Takeaway
The backend now keeps a complete, authoritative record of what the live system observed and decided: every verdict (with request metadata), every non-scored request, and — via a decoupled feedback channel keyed by `X-Verdict-Id` — the true outcome. From that, detection metrics are computed live from the DB. This is the input the **Decide** layer (Phase C2) consumes: windowed rules over the trail that can weight decisions by each subject's historical outcomes. Reproduces `backend/verdict_store.py`, `backend/decision_api.py`, and the middleware + `record_verdict_safe` integration in `backend/app.py`, `identity_api.py`, `ics_api.py`.